In [0]:
# Silver Fraud Indicators

from pyspark.sql import functions as F

CATALOG = "insurance_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

bronze_table = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_fraud_indicators"
silver_table = f"{CATALOG}.{SILVER_SCHEMA}.silver_fraud_indicators"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

fraud_bronze = spark.table(bronze_table)

silver_claims = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_claims")
    .select(F.col("claim_id").alias("valid_claim_id"))
    .dropDuplicates()
)

fraud_prepared = (
    fraud_bronze
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("previous_claims_count", F.col("previous_claims_count").cast("int"))
    .withColumn("suspicious_amount_flag", F.col("suspicious_amount_flag").cast("boolean"))
    .withColumn("duplicate_claim_flag", F.col("duplicate_claim_flag").cast("boolean"))
    .withColumn("late_report_flag", F.col("late_report_flag").cast("boolean"))
    .withColumn("high_risk_region_flag", F.col("high_risk_region_flag").cast("boolean"))
    .withColumn("risk_score", F.col("risk_score").cast("int"))
)

fraud_joined = (
    fraud_prepared
    .join(silver_claims, F.col("claim_id") == F.col("valid_claim_id"), "left")
    .withColumn("claim_exists", F.col("valid_claim_id").isNotNull())
    .drop("valid_claim_id")
)

duplicate_claim_ids = (
    fraud_joined
    .groupBy("claim_id")
    .count()
    .filter(F.col("claim_id").isNotNull() & (F.col("count") > 1))
    .select("claim_id")
    .withColumn("is_duplicate_claim_id", F.lit(True))
)

fraud_checked = (
    fraud_joined
    .join(duplicate_claim_ids, on="claim_id", how="left")
    .withColumn(
        "is_duplicate_claim_id",
        F.coalesce(F.col("is_duplicate_claim_id"), F.lit(False))
    )
)

invalid_condition = (
    F.col("claim_id").isNull()
    | F.col("is_duplicate_claim_id")
    | (~F.col("claim_exists"))
    | F.col("previous_claims_count").isNull()
    | (F.col("previous_claims_count") < 0)
    | F.col("risk_score").isNull()
    | (F.col("risk_score") < 0)
    | (F.col("risk_score") > 100)
    | F.col("suspicious_amount_flag").isNull()
    | F.col("duplicate_claim_flag").isNull()
    | F.col("late_report_flag").isNull()
    | F.col("high_risk_region_flag").isNull()
)

valid_fraud = (
    fraud_checked
    .filter(~invalid_condition)
    .drop("claim_exists", "is_duplicate_claim_id")
    .dropDuplicates(["claim_id"])
)

invalid_fraud_count = fraud_checked.filter(invalid_condition).count()

valid_fraud.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)

print("Bronze fraud indicators:", fraud_bronze.count())
print("Silver fraud indicators:", spark.table(silver_table).count())
print("Invalid fraud indicators excluded from Silver:", invalid_fraud_count)